# Leak-Free Preprocessing Pipelines + k-NN

By the end of this session, you should be able to explain **why preprocessing must be learned from training data only**, and use a scikit=learn `Pipeline` so preprocessing happens correctly inside cross-validation.

We'll use **k-nearest neighbours (K-NN)** because it gives a very clear reason why feature scaling matters without requiring a large amount of new mathematics.

## 1. Retrieval

1. Why do we perform model selection using cross-validation on X_train rather than comparing candidate models using X_test?

> The test set should not influence which model is chosen as it should be an independent estimate of how your chosen modelling process performs on unseen data. Therefore the model should should be chosen based on its performance using X_train and only after this point should be evaluated once on the untouched test set.

2. In 5-fold cross-validation, how many times is each training observation used for validation, and how many times for fitting?

> Each training observation is used four times for fitting and exactly once for validation across the five cross-validation runs.

3. What useful question does a dummy baseline answer?

> The dummy model tells us how well could we perform without learning any useful relationship between the features and the target.

4. Suppose one feature ranges from roughly $0$–$1$ and another from $0$–$10{,}000$. From what you already know about scaling, what potential problem might arise if an algorithm uses distance between observations?

> a feature with a much larger numerical scale can dominate the distance calculation, making differences in smaller-scale features contribute very little even if they are predictively important

## 2. k-nearest neighbour

**k-nearest neighbours (k-NN)** is a classification algorihm with a very simple idea.

Suppose we have a new observation $\mathbf{x}$ whose class is unknown.

We:
1. calculate it's distance to the training observations;
2. find the $k$ closest training observations;
3. look at their class labels;
4. predict the most common class among those neighbours.

For example, with: $$k=5$$ suppose the five closest training observations have labes: $$[1,1,0,1,0]$$

Then k-NN predicts: $$\hat{y}=1$$ because class 1 has three of the five votes.

There isn't a conventional model-fitting stage where k-NN learns coefficients like linear regression or estimates Gaussian parameters like Naive Bayes. It essentially **stores the training examples and uses them when making predictions**.

That's why it's sometimes calles a *lazy learner*.

### Why scaling matters
Because neighbour selection depends directly on distance: $$d(\mathbf{x},\mathbf{z})=\sqrt{\sum_j(x_j-z_j)^2}$$

features with large numerical scales can dominate.

The Breast Cancer Wisconsin dataset contains measurement on quite differen numerical scales, so raw k-NN may trat some features as far more important simply because of their units.

One common solution is **standardisation**.

For each feature: $$z=\frac{x-\mu}{\sigma}$$

After standardisation, the training feature has approximatley: $$\mu=0,\quad\sigma=1$$

So different features becom mush more comparable in scale.

This is the same transformation behind the z-scores you've already covered; here we're using it as **preprocessing for an ML model**.

## 3. Where leakage enters

Suppose we want to perform 5-fold CV. It might seem reasonable to do this first:

$$\mu_j=\text{mean of feature }j\text{ across all }X_{train}\\\sigma_j=\text{SD across all }X_{train}$$

then standardise all of `X_train`, and *afterwards* perform CV.

But imagine fold 1 is currently our validation fold. The scalers $\mu$ and $\sigma$ were calculated using **fold 1 itself**.

So information from the validation observations has influenced the representation of the fitting data.

**That's leakage**.

The correct process for each CV run is:

$$\text{4 fitting folds}\rarr\text{estimate }\mu,\sigma$$

Then use those fitted parameters to transform: $$4\text{ fitting folds}$$ and separatley: $$1\text{ validation fold}$$

The validaiton fold is **transformed using parameters learned elsewhere**; it does not help determine those parameters.

This is the exact same principle you've already been applying:

> Anything learned from data must be learned only from the data available to the model at that stage

## 4. Pipeline

Doing this correclty by hand for every CV fold would be annoying.

A scikit-learn `Pipeline` packages a sequence like: $$\text{StandardScaler}\rarr \text{k-NN}$$ into one estimator.

Then during cross-validation, scikit-learn treats that whole pipeline as the model.

For each fold it:

$$\text{fits scaler on fitting folds}\\ \darr\\ \text{transforms fitting folds} \\ \darr\\ \text{fits k-NN}$$

and then: $$\text{uses the same fitted scaler on validation fold}\rarr \text{k-NN prediction}$$

So pipelines aren't merely a convenient way of making code shorter. They are an important tool for **preventing preprocessing leakage**.

## 5. Conceptual check

1. Suppose you standardise all of X_train before calling 5-fold cross-validation. Why is that data leakage even though X_test has never been touched?

> Because information from the validation fold observations of X-train has influenced the representation of the fitting data.
 
2. Why would you expect standardisation to matter much more for k-NN than it did for Gaussian Naive Bayes?

> Guassian Naive Bayes calucaltes the probability density using:
>
> $$p(x\mid C)=\frac{1}{\sqrt{2\pi\sigma_C^2}}\exp\left(-\frac{(x-\mu_C)^2}{2\sigma_C^2}\right)$$
>
> which already effectivley standardises the observation relative to its mean and standard deviation through the exponent of: $$-\frac{(x-\mu_C)^2}{2\sigma_C^2}$$ k-NN on the other hand calculates raw numerical distance between obervation features as a classification method therefore the larger numerical scales will dominate this calculation without standardisation.

## 6. scaled vs unscaled k-NN

Continue using the same Breast Cancer Wisconsin dataset and the same 80/20 train/test setup. 

Use: $$k=5$$ throughout. We are **not** tuning $k$ today; the point is to isolate preprocessing.

Using **only the training data for model comparison**:
1. Create an ordinary 5-nearest-neighbours classifier with no scaling.
2. Create a second 5-nearest-neighbours classifier where StandardScaler is applied through a scikit-learn pipeline.
3. Evaluate both using 5-fold cross-validation with accuracy.
4. For each, report:
    - the five CV scores;
    - mean CV accuracy;
    - standard deviation.
5. Select the better approach based only on its CV results.
6. Fit the selected approach to the whole training set.
7. Evaluate it once on the untouched test set.

In [28]:
import numpy as np

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

breast_cancer = load_breast_cancer()

X = breast_cancer.data
y = breast_cancer.target

X_train, X_test, y_train, y_test = train_test_split(
  X,
  y, 
  test_size=0.2,
  random_state=42,
  stratify=y
)

# 5-nearest neighbours classifier with no scaling
neigh_raw = KNeighborsClassifier(n_neighbors=5)

# 5-nearest neighbours classifier with StandardScaler
neigh_scaled = Pipeline([
  ('scaler', StandardScaler()),
  ('classifier', KNeighborsClassifier(n_neighbors=5))
])

# Evaluate both using 5-fold CV with accuracy
neigh_raw_CV_scores = cross_val_score(neigh_raw, X_train, y_train, cv=5)
neigh_scaled_CV_scores = cross_val_score(neigh_scaled, X_train, y_train, cv=5)

# Cross validation reporting
labels = [
  '5-nearest-neighbours no scaling CV', 
  '5-nearest-neighbours with scaling CV'
]

for index, scores in enumerate([neigh_raw_CV_scores, neigh_scaled_CV_scores]):
  print(f'\n{labels[index]} results:')
  print(f'accuracy scores: {scores}')
  print(f'mean: {np.mean(scores): .3f}')
  print(f'std: {np.std(scores): .3f}')

# Choose a model based on CV results
print(
  '\nBased on CV results, 5-nearest-neighbours performs noticeably better with StandardScaler() applied than without:\n  - 0.967 mean accuracy with scaling;\n  - 0.936 mean accuracy without scaling.',
  '\n\nSelecting 5-nearest-neighbours with scaling for whole training set fitting'
)

neigh_scaled.fit(X_train, y_train)


# Evaluate once on the test set
neigh_scaled_pred = neigh_scaled.predict(X_test)

n_correct_predictions = np.count_nonzero(neigh_scaled_pred == y_test)
accuracy = n_correct_predictions / len(neigh_scaled_pred)

print(f'\nFinal test accuracy: {accuracy: .3f}')



5-nearest-neighbours no scaling CV results:
accuracy scores: [0.96703297 0.93406593 0.91208791 0.92307692 0.94505495]
mean:  0.936
std:  0.019

5-nearest-neighbours with scaling CV results:
accuracy scores: [0.94505495 1.         0.94505495 0.97802198 0.96703297]
mean:  0.967
std:  0.021

Based on CV results, 5-nearest-neighbours performs noticeably better with StandardScaler() applied than without:
  - 0.967 mean accuracy with scaling;
  - 0.936 mean accuracy without scaling. 

Selecting 5-nearest-neighbours with scaling for whole training set fitting

Final test accuracy:  0.956


Write a short interpretation answering:
    - Did scaling materially affect k-NN?
    - Why does the result make sense given how k-NN works?
    - Why is the pipeline important rather than scaling X_train before cross-validation?

> 1. Scaling had a noticeable effect on the cross validation result scores, with the scaled 5-nearest-neighbours classifier achieving a CV mean accuracy of 0.967 compared to 0.936 without scaling.

> 2. It makes sense that the classifier becomes **more** accurate when feature scaling is applied because the k-NN classifier uses raw numerical distance to classify observations into their respective classes, and since this method biases features with larger numerical scales, standardising all features removes this scaling bias and can therefore lead to more appropriate neighbour selection and better classification performance.

> 3. The pipeline is important because during each cross-validation run the `StandardScaler` must be fitted only on the four fitting folds, while both the fitting folds and validation fold are transformed using those fitted parameters. Passing the scaler and classifier together as a piepline to `cross_val_score` ensures this happens independently for each fold and prevents preprocessing leakage.